# Binomial Option Pricer Visualization

## Importing The Stuff
Start importing the appropriate stuff

In [ ]:
from pathlib import Path
import sys

# Get the directory of the notebook, navigate up, and find 'src'
src_path = Path.cwd().parent / "src"

# Add to system path if it's not already there
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

# Import the other modules
import numpy as np 
import matplotlib.pyplot as plt
from scipy.special import binom

# The most important one
from binomial_option_pricer import binomial_option_pricer, call_payoff, put_payoff

## Inputting Parameters
Input your own parameters below:

- $S$: Spot price of stock
- $K$: Strike price of option
- $T$: Time span
- $N$: Number of time steps
- $\sigma$: Annualized Volatility of stock
- $r$: Continuous market risk-free rate
- call: True if this is a call option, False if put option
- european: True if this is a European-style option, False if American-style

In [ ]:
S = 250
K = 250
T = 10

# Recommend to keep under 30 for legibility
N = 10

sigma = 0.1
r = 0.02
call = True
european = False

### Checking the No-Arbitrage Rule
Make sure to run the following code to see if your inputted parameters violate the No-Arbitrage rule. If all is good, then continue on. If not, then tweak your parameters.

Note that we are using the Cox-Ross-Rubinstein (CRR) parameterization for the up and down factors $u$ and $d$: $u = e^{\sigma \sqrt{dt}}, d=\frac{1}{u}=e^{-\sigma \sqrt{dt}}$ with $dt=T/N$.

In [ ]:
R = np.exp(r*T/N)
u = np.exp(sigma*np.sqrt(T/N))
d = 1/u

if R <= d:
    raise ValueError("Arbitrage oppotunity (R <= d): Borrow money at the market risk-free rate and put it into the stock")
elif R >= u:
    raise ValueError("Arbitrage oppotunity (R >= u): Short-sell the stock and put that money in an investment returning the market risk-free rate")
else:
    print("No-Arbitrage Rule not violated!")


## Tree Visualization
The following block of code will plot the tree. For American-style options, nodes that are highlighted in red represents states where early exercising of the option is most optimal.

In [ ]:
A = binomial_option_pricer(S=S, K=K, T=T, N=N, sigma=sigma, r=r, call=call, european=european)
tree = A.tree

tree_data = np.concatenate([tree[i,:i+1] for i in range(N+1)])
mark_nodes = np.concatenate([A.marked_nodes[i,:i+1] for i in range(N+1)])
Y_data = np.concatenate([[2*j-i for j in range(i+1)] for i in range(N+1)])
X_data = np.concatenate([np.full(i,i-1) for i in range(1,N+2)])

plt.style.use('dark_background')
plt.figure(figsize=(8, 6))

edgecolors = ["none" if mark_nodes[i] == 0 else "red" for i in range(((N+1)*(N+2))//2)]

scatter = plt.scatter(X_data,Y_data,c=tree_data, cmap='viridis',alpha=0.9,edgecolors=edgecolors)
colorbar = plt.colorbar(scatter)
colorbar.set_label("Option Value")
plt.xlabel("Number of time steps")
plt.ylabel("Number of up moves")
plt.grid(False)
plt.show()

print(f"Starting price for option: {A.V0}")